# EgyptGPT Autoresearch

Sets up and runs an autonomous AI research loop on this repo using Claude Code.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**What this does:** Claude Code reads `program.md`, then autonomously edits the model/training code,
trains for 5 minutes, evaluates, keeps improvements, discards regressions, and repeats forever.

**Requirements:** Colab GPU runtime, Anthropic API key.

## 1. Setup

In [ ]:
# Clone repo and install Python dependencies
!git clone https://github.com/JLansey/EgyptGPT.git
%cd EgyptGPT
!bash setup.sh

In [ ]:
# Prepare the character-level dataset (generates train.bin and val.bin)
!python data/egypt_char/prepare.py

In [ ]:
# Verify data is ready
import os
for f in ['data/egypt_char/train.bin', 'data/egypt_char/val.bin', 'data/egypt_char/meta.pkl']:
    size = os.path.getsize(f) if os.path.exists(f) else 'MISSING'
    print(f'{f}: {size}')

## 2. Set your Anthropic API key

You can either:
- Use Colab Secrets (recommended): Add `ANTHROPIC_API_KEY` in the key icon on the left sidebar
- Or paste it directly in the cell below

In [ ]:
import os

# Option A: Load from Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Loaded API key from Colab Secrets')
except Exception:
    pass

# Option B: Paste directly (uncomment and fill in)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set!'
print('API key is set.')

In [ ]:
# Install Node.js and Claude Code
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!npm install -g @anthropic-ai/claude-code
!claude --version

In [ ]:
# Install tmux for session persistence
!sudo apt-get install -y tmux

## 3. Configure git (for the experiment commits)

In [ ]:
# Mount Google Drive for persistent results logging
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for this project's autoresearch results
import os
drive_results_dir = '/content/drive/MyDrive/EgyptGPT_autoresearch'
os.makedirs(drive_results_dir, exist_ok=True)
print(f'Results will be backed up to: {drive_results_dir}')

In [ ]:
!git config user.email "autoresearch@colab"
!git config user.name "autoresearch"

import subprocess, os

# Kill any existing session
subprocess.run(['tmux', 'kill-session', '-t', 'autoresearch'], capture_output=True)

# Build the prompt
prompt = """Read program.md in this directory. This is an autoresearch setup.

1. Create branch autoresearch/run1 from master
2. Verify data/egypt_char/train.bin and val.bin exist
3. Initialize results.tsv with the header row
4. Begin the experiment loop as described in program.md

After each experiment, copy results.tsv to /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv so the log survives session crashes.

Start with the baseline run, then iterate autonomously. Never stop."""

# Start tmux session with Claude Code
cmd = f'''cd /content/EgyptGPT && ANTHROPIC_API_KEY={os.environ["ANTHROPIC_API_KEY"]} claude -p {repr(prompt)} --dangerously-skip-permissions'''
subprocess.run(['tmux', 'new-session', '-d', '-s', 'autoresearch', cmd])

print('Autoresearch is running in tmux session.')
print()
print('To watch:  open Colab terminal, run: tmux attach -t autoresearch')
print('To check:  !cat /content/EgyptGPT/results.tsv')
print('To stop:   !tmux kill-session -t autoresearch')
print()
print('Results backed up to Google Drive: /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv')

In [ ]:
import subprocess, os

# Kill any existing session
subprocess.run(['tmux', 'kill-session', '-t', 'autoresearch'], capture_output=True)

# Build the prompt
prompt = """Read program.md in this directory. This is an autoresearch setup.

1. Create branch autoresearch/run1 from master
2. Verify data/egypt_char/train.bin and val.bin exist
3. Initialize results.tsv with the header row
4. Begin the experiment loop as described in program.md

Start with the baseline run, then iterate autonomously. Never stop."""

# Start tmux session with Claude Code
cmd = f'''cd /content/EgyptGPT && ANTHROPIC_API_KEY={os.environ["ANTHROPIC_API_KEY"]} claude -p {repr(prompt)} --dangerously-skip-permissions'''
subprocess.run(['tmux', 'new-session', '-d', '-s', 'autoresearch', cmd])

print('Autoresearch is running in tmux session.')
print()
print('To watch:  open Colab terminal, run: tmux attach -t autoresearch')
print('To check:  !cat /content/EgyptGPT/results.tsv')
print('To stop:   !tmux kill-session -t autoresearch')

## 5. Monitor progress

In [ ]:
# Check results so far
!cat /content/EgyptGPT/results.tsv 2>/dev/null || echo 'No results yet.'

In [ ]:
# Check git log for kept experiments
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
# Check if autoresearch is still running
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'